# Practice Lab: The AI Gateway Pattern (Lesson 12.3)

Module 12 . 4 exercises . Model registry + budgets + semantic cache + production deploy


# Lesson 12.3: The AI Gateway Pattern

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import numpy as np
import hashlib
np.random.seed(42)


---
## Exercise 1 (Easy): Add Model to Gateway


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class GatewayModels:
    def __init__(self):
        self.models={"gemini-2.0-flash":{"in":0.00015,"out":0.0006},"gemini-2.5-pro":{"in":0.00125,"out":0.005},
            "gpt-4o":{"in":0.0025,"out":0.01},"claude-sonnet-4":{"in":0.003,"out":0.015}}
        self.usage={m:{"calls":0,"cost":0.0} for m in self.models}
    def add_model(self,name,inp,out):
        self.models[name]={"in":inp,"out":out}; self.usage[name]={"calls":0,"cost":0.0}
    def call(self,model,prompt,max_tok=500):
        p=self.models[model]; ti=max(1,int(len(prompt.split())*1.3)); to=min(max_tok,ti+50)
        cost=ti/1000*p["in"]+to/1000*p["out"]; self.usage[model]["calls"]+=1; self.usage[model]["cost"]+=cost
        return {"model":model,"cost":round(cost,6),"tokens":ti+to}

gw=GatewayModels()
gw.add_model("claude-haiku-3.5",inp=0.0008,out=0.004)
print("Gateway - Add Model:")
reqs=[("gemini-2.0-flash","Course price?"),("claude-haiku-3.5","Refund policy"),
    ("gemini-2.5-pro","Analyze Indian GenAI market"),("gpt-4o","Write marketing email"),
    ("claude-sonnet-4","Design RAG pipeline"),("gemini-2.0-flash","List modules"),
    ("claude-haiku-3.5","Prerequisites?"),("gemini-2.0-flash","Hello"),
    ("claude-haiku-3.5","Explain embeddings"),("gpt-4o","Compare Python vs JS")]
for model,prompt in reqs:
    r=gw.call(model,prompt); print(f"  {r['model']:<22} {r['tokens']:>5} tok  ${r['cost']:.6f}")
print(f"\n  Per-Model:")
for m,s in sorted(gw.usage.items(),key=lambda x:gw.models[x[0]]["in"]):
    if s["calls"]>0: print(f"  {m:<22} calls={s['calls']} cost=${s['cost']:.6f} rate=${gw.models[m]['in']*1000:.2f}/MTok")


</details>


---
## Exercise 2 (Easy): Budget Alerts


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class BudgetAlerts:
    def __init__(self): self.teams={}; self.spending={}
    def add(self,name,budget): self.teams[name]=budget; self.spending[name]=0.0
    def spend(self,team,amt):
        if self.spending[team]>=self.teams[team]: return "BLOCKED"
        self.spending[team]+=amt; return "OK"
    def alerts(self):
        result=[]
        for t,b in self.teams.items():
            s=self.spending[t]; pct=s/b*100 if b>0 else 0
            lvl="BLOCKED" if pct>=100 else "CRITICAL" if pct>=90 else "WARNING" if pct>=75 else "INFO" if pct>=50 else "OK"
            result.append({"team":t,"budget":b,"spent":round(s,2),"pct":round(pct,1),"level":lvl,"left":round(b-s,2)})
        return sorted(result,key=lambda x:-x["pct"])

ba=BudgetAlerts()
ba.add("Engineering",100); ba.add("Marketing",30); ba.add("Support",50); ba.add("Research",20)
for _ in range(40): ba.spend("Engineering",2.0)
for _ in range(25): ba.spend("Marketing",1.2)
for _ in range(30): ba.spend("Support",1.0)
for _ in range(18): ba.spend("Research",1.1)
print("Budget Alerts:")
print(f"  {'Team':<14} {'Budget':>7} {'Spent':>7} {'%':>6} {'Left':>7}  Level")
for a in ba.alerts():
    print(f"  {a['team']:<14} ${a['budget']:>5.0f} ${a['spent']:>5.1f} {a['pct']:>5.1f}% ${a['left']:>5.1f}  {a['level']}")
crit=[a for a in ba.alerts() if a["level"] in ("CRITICAL","BLOCKED")]
print(f"\n  Action needed: {len(crit)} team(s)")


</details>


---
## Exercise 3 (Medium): Semantic Cache


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib; np.random.seed(42)

class SemCache:
    def __init__(self,th=0.80): self.exact={}; self.sem=[]; self.th=th; self.stats={"exact":0,"semantic":0,"miss":0}; self.saved=0.0
    def _h(self,t): return hashlib.md5(t.lower().strip().encode()).hexdigest()
    def _emb(self,t):
        v=np.zeros(32)
        for w in t.lower().split(): v[hash(w)%32]+=1.0
        n=np.linalg.norm(v); return v/n if n>0 else v
    def get(self,prompt,cost=0.003):
        k=self._h(prompt)
        if k in self.exact: self.stats["exact"]+=1; self.saved+=cost; return "EXACT",self.exact[k]
        qe=self._emb(prompt); best=0; br=None
        for e,r,_ in self.sem:
            s=float(np.dot(qe,e))
            if s>best: best=s; br=r
        if best>=self.th and br: self.stats["semantic"]+=1; self.saved+=cost; return f"SEM({best:.2f})",br
        self.stats["miss"]+=1; return "MISS",None
    def put(self,p,r,c=0.003): self.exact[self._h(p)]=r; self.sem.append((self._emb(p),r,c))

c=SemCache(0.80)
for p,r in [("What is the course price?","14999 INR"),("What are the prerequisites?","Python basics"),
    ("How long is the course?","146 hours"),("What is the refund policy?","7d full, 30d 50%"),("When do classes start?","July 2026")]:
    c.put(p,r)
print("Semantic Cache:")
for q in ["What is the course price?","How much does it cost?","What are the prerequisites?","Tell me the requirements",
    "Compare GenAI vs Python","How long is the course?","What is the duration?","Design RAG pipeline",
    "What is the refund policy?","Can I get money back?","Explain transformers","When do classes start?",
    "Next batch date?","Deploy on Cloud Run","What is the course price?"]:
    tier,r=c.get(q)
    if r is None: c.put(q,f"Resp:{q[:15]}")
    print(f"  {tier:>15}: {q[:35]}")
total=sum(c.stats.values()); hr=(c.stats["exact"]+c.stats["semantic"])/total*100
print(f"\n  Stats: {c.stats} | Hit rate: {hr:.0f}% | Saved: ${c.saved:.4f}")


</details>


---
## Exercise 4 (Challenge): Production Deploy


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Production Gateway Deploy:")
print("  Architecture: Cloud Run -> Redis -> PostgreSQL -> Grafana")
print("  Dockerfile: python:3.12-slim + uvicorn --workers 4")
print("  Deploy: gcloud run deploy --memory 4Gi --cpu 2 --region asia-south1")
print("\n  Grafana Panels:")
for p in ["Cost/Team (bar)","Cost/Model (pie)","Req/min (line)","Cache Hit% (gauge)",
    "P50/P95 Latency","Error Rate","Budget Util%","Token Trend"]:
    print(f"    - {p}")
costs=[("Cloud Run",40,80),("Redis",35,45),("PostgreSQL",10,20),("Monitoring",0,10)]
tl=sum(l for _,l,_ in costs); th=sum(h for _,_,h in costs)
print(f"\n  Monthly infra: ${tl}-${th} (asia-south1)")
print(f"  + LLM API costs per token")


</details>
